# 训练标签置换对照：模式监督是否带来关联增量

本 Notebook 读取固定公开数据上的 999 次置换结果，核对固定评估复现、任务折和配对哈希，并展示两种候选库协议的主统计量。完整实验由 `scripts/run_mode_label_permutation.py` 重算；PDF/PNG 图由 `scripts/generate_paper_figures.py` 生成。

置换在每个外层折内，仅打乱训练集同一 benchmark/task 组内的 Human/Agent 标签；每个组完整地进入训练或测试。保持关联训练配对及配对标签、查询和候选集合、外层任务折不变，重新拟合嵌套 OOF 模式头和模式感知排序器。观察值是实际标签的 Human→Agent benchmark-macro 配对 Δ Recall@1。历史候选库为主协议，严格留出候选库为稳健性协议。

这是已探索数据上的条件性负对照。模式标签与日志采集路径混杂，因此经验尾部比例不作因果结论或总体显著性检验。

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd()
if (ROOT / 'research' / 'scripts').is_dir(): ROOT = ROOT / 'research'
if not (ROOT / "results").exists():
    ROOT = ROOT.parent
result = json.loads((ROOT / "results/mode_label_permutation.json").read_text())
assert result["fixed_evaluation_reproduction"]["status"] == "passed"
assert result["failed_permutations"] == 0
assert result["protocol"]["permutations"] >= 999
print(f"{result['data']['records']} records · {result['data']['benchmark_task_groups']} task groups · {result['protocol']['permutations']} permutations")
print("fixed evaluation:", result["fixed_evaluation_reproduction"])

In [ ]:
rows = []
for key, label in [("full_gallery", "Historical gallery · primary"),
                   ("heldout_gallery", "Held-out-only gallery · secondary")]:
    item = result["permutation_summary"][key]
    lo, hi = item["permuted_delta_95_reference_interval"]
    rows.append({
        "Protocol": label,
        "Tasks": result["observed"][key]["task_count"],
        "Observed Δ (pp)": 100 * item["observed_mode_minus_similarity_delta"],
        "Permuted mean Δ (pp)": 100 * item["permuted_delta_mean"],
        "95% reference interval (pp)": f"[{100*lo:.2f}, {100*hi:.2f}]",
        "Upper-tail count / B": f"{item['permutations_at_or_above_observed']} / {item['permutation_count']}",
        "Empirical upper-tail estimate": item["empirical_upper_tail_fraction"],
    })
summary = pd.DataFrame(rows)
display(summary.style.format({
    "Observed Δ (pp)": "{:+.2f}",
    "Permuted mean Δ (pp)": "{:+.2f}",
    "Empirical upper-tail estimate": "{:.4f}",
}).background_gradient(subset=["Observed Δ (pp)"], cmap="PuOr", vmin=-10, vmax=10))

In [ ]:
# Split and pairing integrity: held-out task groups occur exactly once across outer test folds.
folds = result["outer_folds"]
heldout_tasks = [task for fold in folds for task in fold["heldout_task_ids"]]
assert len(heldout_tasks) == len(set(heldout_tasks)) == result["data"]["benchmark_task_groups"]
for fold in folds:
    assert not set(fold["training_task_ids"]) & set(fold["heldout_task_ids"])
    assert len(fold["training_trace_ids"]) + len(fold["heldout_trace_ids"]) == result["data"]["records"]
    expected_pairs = fold["fixed_association_pairs_sha256"]
    for replicate in result["replicates"]:
        audit = next(item for item in replicate["fold_audits"] if item["fold"] == fold["fold"])
        assert audit["fixed_association_pairs_sha256"] == expected_pairs
print("Passed: disjoint task folds, complete held-out task coverage, and fixed association-pair hashes across all permutations.")
print("Environment:", result["software"])
print("Seed:", result["protocol"]["seed"])

In [ ]:
display(Image(filename=str(ROOT / "paper/figures/figureA5_mode_label_permutation.png")))

## Interpretation boundary

A real-label effect outside the shuffled-label reference distribution supports trace-label alignment as a useful signal **within this benchmark composition and candidate protocol**. It does not identify an execution-mode causal effect: human traces are terminal recordings while Agent traces are tool-call records. A benefit that disappears in the held-out-only gallery is protocol-sensitive and cannot yet support a robust activity-correlation claim. Task IDs remain same-challenge proxies rather than independent attack-activity ground truth.